# 🍽️ Foodseekers — Multilingual Prompt-Attack Classification
### RMIT Hackathon 2026 | Track: Security × Generative AI × Low-Resource Languages
**Team Foodseekers (UIT - RMIT Saigon South)**  
Pipeline: Feature-Union (Word + Char-wb TF-IDF) + Balanced Linear Classifier with 5-Fold Stratified Ensemble  
Metric: **Macro-averaged F1 Score** (BENIGN, JAILBREAK, PROMPT_INJECTION)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score

# 1. Đường dẫn dữ liệu (Tự động thích ứng cả máy local và Kaggle)
TRAIN_PATH = 'train.csv' if os.path.exists('train.csv') else ('E:/Hackathon/train.csv' if os.path.exists('E:/Hackathon/train.csv') else '/kaggle/input/rmit-hackathon-2026/train.csv')
TEST_PATH = 'test.csv' if os.path.exists('test.csv') else ('E:/Hackathon/test.csv' if os.path.exists('E:/Hackathon/test.csv') else '/kaggle/input/rmit-hackathon-2026/test.csv')
SUBMISSION_PATH = 'submission_optimized.csv'

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print("\nPhân bố nhãn trên tập huấn luyện:")
print(train_df['label'].value_counts())

In [ ]:
# 2. Xây dựng bộ trích xuất đặc trưng đa ngữ (Word + Sub-word / Character n-grams)
# Trích xuất word n-grams (1, 2) bắt ngữ nghĩa câu và char-wb (3, 5) chống adversarial obfuscation / leetspeak
feature_extractor = FeatureUnion([
    ('word', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=45000,
        sublinear_tf=True,
        strip_accents='unicode',
        token_pattern=r'(?u)\b\w+\b'
    )),
    ('char', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=30000,
        sublinear_tf=True
    ))
])

print("Fitting FeatureUnion trên toàn bộ văn bản...")
X_train_all = feature_extractor.fit_transform(train_df['text'])
X_test = feature_extractor.transform(test_df['text'])

classes = ['BENIGN', 'JAILBREAK', 'PROMPT_INJECTION']
label_to_idx = {c: i for i, c in enumerate(classes)}
y_train_all = np.array([label_to_idx[l] for l in train_df['label']])

print(f"Kích thước ma trận đặc trưng: {X_train_all.shape}")

In [ ]:
# 3. Huấn luyện 5-Fold Stratified Cross-Validation & Ensemble Out-Of-Fold (OOF)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train_df), 3))
test_probs = np.zeros((len(test_df), 3))

fold_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_all, y_train_all)):
    X_tr, y_tr = X_train_all[train_idx], y_train_all[train_idx]
    X_va, y_va = X_train_all[val_idx], y_train_all[val_idx]
    
    # Sử dụng class_weight='balanced' để bảo vệ lớp PROMPT_INJECTION (chỉ 7% mẫu nhưng chiếm 33.3% Macro F1)
    model = LogisticRegression(C=2.5, class_weight='balanced', max_iter=1000, random_state=42 + fold)
    model.fit(X_tr, y_tr)
    
    val_preds_prob = model.predict_proba(X_va)
    oof_probs[val_idx] = val_preds_prob
    
    val_preds = np.argmax(val_preds_prob, axis=1)
    score = f1_score(y_va, val_preds, average='macro')
    fold_scores.append(score)
    print(f"Fold {fold + 1} Macro F1: {score:.4f}")
    
    test_probs += model.predict_proba(X_test) / skf.n_splits

oof_preds = np.argmax(oof_probs, axis=1)
total_macro_f1 = f1_score(y_train_all, oof_preds, average='macro')
print(f"\n>>> TỔNG ĐIỂM 5-FOLD OOF MACRO F1: {total_macro_f1:.4f} <<<")
print("\nBáo cáo phân loại chi tiết từng lớp:")
print(classification_report(y_train_all, oof_preds, target_names=classes, digits=4))

In [ ]:
# 4. Xuất file nộp bài tối ưu (submission_optimized.csv)
final_test_preds_idx = np.argmax(test_probs, axis=1)
final_test_labels = [classes[idx] for idx in final_test_preds_idx]

sub_df = pd.DataFrame({
    'id': test_df['id'],
    'label': final_test_labels
})

sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Đã lưu dự đoán vào: {SUBMISSION_PATH}")
print(f"Số dòng (kể cả header): {len(sub_df) + 1}")
print("\nPhân bố nhãn dự đoán trên tập kiểm thử:")
print(sub_df['label'].value_counts())
print(sub_df['label'].value_counts(normalize=True))

In [ ]:
# 5. Hướng dẫn nộp bài lên Kaggle và các bước tinh chỉnh tiếp theo (Deep Learning Transformer)
print("""
========================================================================================
CHIẾN THUẬT TIẾP THEO ĐỂ TỐI ĐA HÓA ĐIỂM SỐ (> 0.93 - 0.95):
1. Bước 1: Nộp file 'submission_optimized.csv' này lên Kaggle để kiểm tra bước nhảy điểm từ 0.84.
2. Bước 2 (GPU Kaggle): Huấn luyện Fine-tune mô hình Transformer đã được chấp thuận:
   - FacebookAI/xlm-roberta-base (Đa ngữ tốt nhất cho pt, fr, it, en)
   - protectai/deberta-v3-base-prompt-injection-v2 (Chuyên trị phát hiện Prompt Injection)
3. Bước 3: Ensemble mềm (Weighted Soft Voting) giữa xác suất Transformer và TF-IDF 5-Fold.
========================================================================================
""")